# CUDA → ROCm / HIP Migration Lab (MI300)

**Goal:** learn GPU-kernel migration *by doing it* — write a HIP kernel, then take real CUDA
code and port it to ROCm, validating every step against a **golden reference**.

Run these cells **top to bottom on your MI300 instance**. Each step is small and checkable.

> Mental model: **HIP** is ~1:1 with CUDA. "Migrating to ROCm" mostly means **CUDA → HIP**,
> after which the same code runs on AMD (and could still run on NVIDIA).

## 0 · Environment check
Confirm the ROCm toolchain is present and detect your GPU architecture (MI300 = `gfx942`).

In [ ]:
!hipcc --version
!echo "---"
!rocm-smi --showproductname 2>/dev/null || rocm-smi
!echo "---"
!rocminfo | grep -m1 -A1 "Name:" ; rocminfo 2>/dev/null | grep -o "gfx[0-9a-f]*" | head -1

In [ ]:
import subprocess
out = subprocess.run(["bash","-lc","rocminfo | grep -o 'gfx[0-9a-f]*' | head -1"],
                     capture_output=True, text=True)
ARCH = out.stdout.strip() or "gfx942"     # MI300X = gfx942
print("Detected GPU arch:", ARCH)
import os; os.makedirs("work", exist_ok=True); os.chdir("work")
print("Working dir:", os.getcwd())

## 1 · Kernel fundamentals (CUDA vs HIP)

| Concept | CUDA | HIP / AMD |
|---|---|---|
| one thread | thread | work-item |
| group (shares fast mem) | block | workgroup |
| SIMD unit | warp = **32** | wavefront = **64** (CDNA/MI) |
| fast on-chip memory | shared | LDS |
| alloc / copy / free | `cudaMalloc` / `cudaMemcpy` / `cudaFree` | `hipMalloc` / `hipMemcpy` / `hipFree` |
| kernel qualifier | `__global__` | `__global__` (same) |
| launch | `k<<<grid,block>>>(...)` | same, or `hipLaunchKernelGGL(k,grid,block,0,0,...)` |

⚠️ The **wavefront = 64** difference is the #1 porting gotcha: any CUDA code assuming warp size 32
(warp-shuffle reductions, `__ballot`, etc.) needs attention.

## 2 · Your first HIP kernel — vector add
Write it, compile with `hipcc`, run. The program verifies itself on the host.

In [ ]:
%%writefile vadd.hip
#include <hip/hip_runtime.h>
#include <cstdio>
#include <cmath>

__global__ void vadd(const float* a, const float* b, float* c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}

int main() {
    const int n = 1 << 20; const size_t sz = (size_t)n * sizeof(float);
    float *ha=(float*)malloc(sz), *hb=(float*)malloc(sz), *hc=(float*)malloc(sz);
    for (int i=0;i<n;++i){ ha[i]=i*0.001f; hb[i]=2.0f*i*0.001f; }

    float *da,*db,*dc; hipMalloc(&da,sz); hipMalloc(&db,sz); hipMalloc(&dc,sz);
    hipMemcpy(da,ha,sz,hipMemcpyHostToDevice);
    hipMemcpy(db,hb,sz,hipMemcpyHostToDevice);

    int tpb=256, blocks=(n+tpb-1)/tpb;
    vadd<<<blocks,tpb>>>(da,db,dc,n);
    hipDeviceSynchronize();

    hipMemcpy(hc,dc,sz,hipMemcpyDeviceToHost);
    double maxerr=0; for(int i=0;i<n;++i) maxerr=fmax(maxerr, fabs(hc[i]-(ha[i]+hb[i])));
    printf("vadd max_err=%g -> %s\n", maxerr, maxerr<1e-5 ? "PASS":"FAIL");

    hipFree(da);hipFree(db);hipFree(dc); free(ha);free(hb);free(hc);
    return 0;
}

In [ ]:
!hipcc vadd.hip -o vadd --offload-arch={ARCH}
!./vadd

## 3 · The migration — start from CUDA, hipify to HIP

Here is a real **CUDA** program (`saxpy`: `y = a*x + y`). We'll convert it with **`hipify-perl`**
and inspect exactly what changed.

In [ ]:
%%writefile saxpy.cu
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>

__global__ void saxpy(int n, float a, const float* x, float* y) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) y[i] = a * x[i] + y[i];
}

int main() {
    const int n = 1 << 20; const size_t sz = (size_t)n * sizeof(float); const float a = 2.0f;
    float *hx=(float*)malloc(sz), *hy=(float*)malloc(sz);
    for (int i=0;i<n;++i){ hx[i]=1.0f; hy[i]=(float)i; }

    float *dx,*dy; cudaMalloc(&dx,sz); cudaMalloc(&dy,sz);
    cudaMemcpy(dx,hx,sz,cudaMemcpyHostToDevice);
    cudaMemcpy(dy,hy,sz,cudaMemcpyHostToDevice);

    int tpb=256, blocks=(n+tpb-1)/tpb;
    saxpy<<<blocks,tpb>>>(n,a,dx,dy);
    cudaDeviceSynchronize();

    cudaMemcpy(hy,dy,sz,cudaMemcpyDeviceToHost);
    FILE* f=fopen("saxpy_out.bin","wb"); fwrite(hy,sizeof(float),n,f); fclose(f);
    printf("saxpy wrote saxpy_out.bin (n=%d)\n", n);

    cudaFree(dx); cudaFree(dy); free(hx); free(hy);
    return 0;
}

In [ ]:
# Translate CUDA -> HIP and show the diff (what hipify actually changed):
!hipify-perl saxpy.cu > saxpy.hip
!echo "================ DIFF (cuda -> hip) ================"
!diff -u saxpy.cu saxpy.hip || true

What you should see hipify change: `cuda_runtime.h → hip/hip_runtime.h`,
`cudaMalloc → hipMalloc`, `cudaMemcpy → hipMemcpy`, `cudaMemcpyHostToDevice → hipMemcpyHostToDevice`,
`cudaDeviceSynchronize → hipDeviceSynchronize`, `cudaFree → hipFree`. The kernel body is unchanged —
that's the point: HIP is a strict subset-compatible mirror of the CUDA runtime.

In [ ]:
!hipcc saxpy.hip -o saxpy --offload-arch={ARCH}
!./saxpy

## 4 · Golden-reference validation (the verification mindset)
Compute the result independently in NumPy and compare — this is exactly how a CI suite asserts
correctness of a migrated kernel.

In [ ]:
import numpy as np
n = 1 << 20; a = 2.0
x = np.ones(n, np.float32); y0 = np.arange(n, np.float32)
golden = (a * x.astype(np.float64) + y0.astype(np.float64))
got = np.fromfile("saxpy_out.bin", np.float32)
err = float(np.max(np.abs(got - golden)))
print("saxpy max abs err vs golden:", err,
      "->", "PASS" if np.allclose(got, golden, rtol=1e-4, atol=1e-2) else "FAIL")

## 5 · A real kernel — row-wise softmax (shared memory + reduction)

This CUDA kernel uses **shared memory** and a **tree reduction**. Note it uses `__syncthreads()`
(not warp-shuffle), so it is **wavefront-size agnostic** — it ports cleanly to AMD's 64-wide wavefronts.

In [ ]:
%%writefile softmax.cu
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>
#include <cmath>
#define COLS 256
__global__ void softmax_row(const float* in, float* out, int cols) {
    int row = blockIdx.x, t = threadIdx.x;
    __shared__ float buf[COLS];
    float v = (t < cols) ? in[row*cols+t] : -1e30f;
    buf[t]=v; __syncthreads();
    for (int s=blockDim.x/2; s>0; s>>=1){ if(t<s) buf[t]=fmaxf(buf[t],buf[t+s]); __syncthreads(); }
    float m=buf[0]; __syncthreads();
    float e = (t<cols) ? expf(v-m) : 0.0f;
    buf[t]=e; __syncthreads();
    for (int s=blockDim.x/2; s>0; s>>=1){ if(t<s) buf[t]+=buf[t+s]; __syncthreads(); }
    float sum=buf[0];
    if (t<cols) out[row*cols+t]=e/sum;
}
int main() {
    const int rows=1024, cols=COLS; const size_t sz=(size_t)rows*cols*sizeof(float);
    float *hin=(float*)malloc(sz), *hout=(float*)malloc(sz);
    for (int i=0;i<rows*cols;++i) hin[i]=((i*37)%100)/50.0f - 1.0f;
    float *din,*dout; cudaMalloc(&din,sz); cudaMalloc(&dout,sz);
    cudaMemcpy(din,hin,sz,cudaMemcpyHostToDevice);
    softmax_row<<<rows,cols>>>(din,dout,cols);
    cudaDeviceSynchronize();
    cudaMemcpy(hout,dout,sz,cudaMemcpyDeviceToHost);
    FILE* g=fopen("softmax_in.bin","wb");  fwrite(hin, sizeof(float),(size_t)rows*cols,g); fclose(g);
    FILE* f=fopen("softmax_out.bin","wb"); fwrite(hout,sizeof(float),(size_t)rows*cols,f); fclose(f);
    printf("softmax wrote softmax_in.bin / softmax_out.bin (%dx%d)\n", rows, cols);
    cudaFree(din); cudaFree(dout); free(hin); free(hout);
    return 0;
}

In [ ]:
!hipify-perl softmax.cu > softmax.hip
!hipcc softmax.hip -o softmax --offload-arch={ARCH}
!./softmax

In [ ]:
import numpy as np
rows, cols = 1024, 256
inp = np.fromfile("softmax_in.bin",  np.float32).reshape(rows, cols).astype(np.float64)
got = np.fromfile("softmax_out.bin", np.float32).reshape(rows, cols)
z = inp - inp.max(1, keepdims=True); e = np.exp(z); gold = e / e.sum(1, keepdims=True)
err = float(np.max(np.abs(got - gold)))
print("softmax max abs err vs golden:", err, "->", "PASS" if err < 1e-5 else "FAIL")

## 6 · Computer-vision kernel — RGB → grayscale (2D grid)

A classic OpenCV operation as a GPU kernel. This one shows **2D** grids/blocks (`dim3`) and
one thread per pixel. We make a test image in NumPy (or point it at your own OpenCV sample),
run the kernel, and validate against the standard luma golden reference.

In [ ]:
import numpy as np
# --- Test image: synthetic by default (no dataset needed for a correctness test).
#     To use YOUR OpenCV sample instead, replace the block below with:
#         import cv2
#         img = cv2.cvtColor(cv2.imread('your_image.jpg'), cv2.COLOR_BGR2RGB)
#         H, W = img.shape[:2]
H, W = 1080, 1920
yy, xx = np.mgrid[0:H, 0:W]
R = ((xx * 255) // W).astype(np.uint8)
G = ((yy * 255) // H).astype(np.uint8)
B = (((xx + yy) * 255) // (W + H)).astype(np.uint8)
img = np.dstack([R, G, B]).astype(np.uint8)     # H x W x 3, RGB
img.tofile("image_rgb.bin")
print("wrote image_rgb.bin", img.shape, img.dtype)

In [ ]:
%%writefile grayscale.cu
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>
__global__ void rgb2gray(const unsigned char* rgb, float* gray, int W, int H) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x < W && y < H) {
        int i = (y * W + x) * 3;
        gray[y * W + x] = 0.299f*rgb[i] + 0.587f*rgb[i+1] + 0.114f*rgb[i+2];
    }
}
int main(int argc, char** argv) {
    int W = argc>1?atoi(argv[1]):1920, H = argc>2?atoi(argv[2]):1080;
    size_t npix=(size_t)W*H;
    unsigned char* hrgb=(unsigned char*)malloc(npix*3);
    FILE* f=fopen("image_rgb.bin","rb"); fread(hrgb,1,npix*3,f); fclose(f);
    float* hgray=(float*)malloc(npix*sizeof(float));
    unsigned char* drgb; float* dgray;
    cudaMalloc(&drgb,npix*3); cudaMalloc(&dgray,npix*sizeof(float));
    cudaMemcpy(drgb,hrgb,npix*3,cudaMemcpyHostToDevice);
    dim3 blk(16,16), grd((W+15)/16,(H+15)/16);
    rgb2gray<<<grd,blk>>>(drgb,dgray,W,H);
    cudaDeviceSynchronize();
    cudaMemcpy(hgray,dgray,npix*sizeof(float),cudaMemcpyDeviceToHost);
    FILE* o=fopen("gray_out.bin","wb"); fwrite(hgray,sizeof(float),npix,o); fclose(o);
    printf("grayscale wrote gray_out.bin (%dx%d)\n", W, H);
    cudaFree(drgb); cudaFree(dgray); free(hrgb); free(hgray);
    return 0;
}

In [ ]:
!hipify-perl grayscale.cu > grayscale.hip
!hipcc grayscale.hip -o grayscale --offload-arch={ARCH}
!./grayscale 1920 1080

In [ ]:
import numpy as np
H, W = 1080, 1920
img = np.fromfile("image_rgb.bin", np.uint8).reshape(H, W, 3).astype(np.float64)
golden = 0.299*img[:,:,0] + 0.587*img[:,:,1] + 0.114*img[:,:,2]
got = np.fromfile("gray_out.bin", np.float32).reshape(H, W)
err = float(np.max(np.abs(got - golden)))
print("grayscale max abs err vs golden:", err, "->", "PASS" if err < 1e-2 else "FAIL")
# Optional cross-check against OpenCV, if you have it installed:
try:
    import cv2
    cvg = cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float64)
    print("vs OpenCV cvtColor (rounded to uint8) max abs err:", float(np.max(np.abs(got - cvg))))
except Exception as e:
    print("(OpenCV not installed - skipped the cv2 cross-check)")

## 7 · (Optional) Benchmark CUDA-vs-ROCm style timing
Time the HIP kernel with HIP events — the same number you'd track as a perf baseline in CI.

In [ ]:
%%writefile bench.hip
#include <hip/hip_runtime.h>
#include <cstdio>
__global__ void saxpy(int n,float a,const float*x,float*y){int i=blockIdx.x*blockDim.x+threadIdx.x; if(i<n) y[i]=a*x[i]+y[i];}
int main(){
    const int n=1<<24; size_t sz=(size_t)n*sizeof(float); float a=2.0f;
    float *x,*y; hipMalloc(&x,sz); hipMalloc(&y,sz); hipMemset(x,0,sz); hipMemset(y,0,sz);
    int tpb=256, blocks=(n+tpb-1)/tpb;
    hipEvent_t s,e; hipEventCreate(&s); hipEventCreate(&e);
    saxpy<<<blocks,tpb>>>(n,a,x,y); hipDeviceSynchronize();      // warmup
    hipEventRecord(s);
    for(int k=0;k<100;++k) saxpy<<<blocks,tpb>>>(n,a,x,y);
    hipEventRecord(e); hipEventSynchronize(e);
    float ms=0; hipEventElapsedTime(&ms,s,e);
    double gbps = 100.0 * 3.0 * sz / (ms/1e3) / 1e9;   // 2 reads + 1 write
    printf("saxpy n=%d: %.3f ms/100 iters, ~%.1f GB/s\n", n, ms, gbps);
    hipFree(x); hipFree(y); return 0;
}

In [ ]:
!hipcc bench.hip -o bench --offload-arch={ARCH}
!./bench

## 8 · (Optional) Migrate a PyTorch custom CUDA extension — auto-hipify

PyTorch's `CUDAExtension` **auto-hipifies** CUDA sources on a ROCm build, so a custom op ports
with (almost) no manual edits. Requires a ROCm build of PyTorch on this box.

```bash
python -c "import torch; print(torch.__version__, torch.version.hip)"   # expect a HIP version
git clone https://github.com/pytorch/extension-cpp
cd extension-cpp && pip install -e . -v     # auto-hipifies .cu -> .hip under the hood
python -m pytest -q                         # or follow the repo README's test command
```
This is your bridge from raw-kernel migration to framework-level migration.

## 9 · Gotchas cheat-sheet (keep this handy)

| CUDA construct | ROCm / HIP note |
|---|---|
| warp size 32 | **wavefront 64** on CDNA/MI — fix reductions/`__shfl_*`/`__ballot` |
| inline **PTX** `asm` | no PTX on AMD — rewrite in HIP/intrinsics |
| Tensor Cores (WMMA) | AMD **MFMA / WMMA** intrinsics (or use rocWMMA / Composable Kernel) |
| cuBLAS / cuDNN / NCCL | **rocBLAS·hipBLAS / MIOpen / RCCL** |
| Thrust / CUB | **rocThrust / hipCUB** |
| `nvcc`, `-arch=sm_XX` | `hipcc`, `--offload-arch=gfx942` (MI300) |
| `cudaMallocAsync`, streams | HIP equivalents exist (`hipMallocAsync`, `hipStream_t`) |

**The verification principle (your north star):** a migrated kernel is "done" only when it matches
the **golden reference** within tolerance — correctness is asserted by the reference, never assumed
from "it compiled."

### Next: `tspeterkim/flash-attention-minimal`
Raw CUDA flash attention, no official ROCm port — hipify it, build with `hipcc`, and validate
against PyTorch `scaled_dot_product_attention` as the golden reference. That's a portfolio-worthy migration.